In [2]:
import os
from typing import Optional, List
import pandas as pd

# Download data from Kaggle and place in a directory, then use the function below to load and filter by date range.
# https://www.kaggle.com/datasets/frankcaoyun/stocktwits-2020-2022-raw

# Loading all CSV files in a directory (and subdirectories), filtering for rows where the `created_at` column is within a date range, and 
# optionally saving the combined results to a new CSV file. 
# This function is memory-efficient by processing files in chunks and streaming results to disk if a save path is provided.
def load_date_range_data_from_dir(root_dir: str, start_date: str, end_date: str, created_at_col: str = 'created_at', chunksize: int = 100_000, save_path: Optional[str] = None) -> pd.DataFrame:
    """Recursively read all .csv files under `root_dir`, keeping rows where `created_at` is within [start_date, end_date].

    Parameters:
    - root_dir: path to directory containing csv files (recursed).
    - start_date: inclusive range start, for example '2022-01-01'.
    - end_date: inclusive range end, for example '2022-03-31'.
    - created_at_col: name of the timestamp column in the csvs.
    - chunksize: rows per chunk when reading each csv (memory-safe).
    - save_path: optional file path to write the combined rows as a csv. If provided, rows are streamed to that file to avoid keeping everything in memory.

    Returns:
    - pd.DataFrame containing all rows in the date range (unless `save_path` used, in which case an empty DataFrame is returned and the combined CSV is written).
    """
    start_ts = pd.to_datetime(start_date, utc=True)
    end_ts = pd.to_datetime(end_date, utc=True) + pd.Timedelta(days=1)
    if start_ts >= end_ts:
        raise ValueError('start_date must be before or equal to end_date')

    csv_files: List[str] = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith('.csv'):
                csv_files.append(os.path.join(dirpath, fn))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found under {root_dir}')

    # helper to filter a chunk for the requested date range
    def _filter_chunk(df: pd.DataFrame) -> pd.DataFrame:
        if created_at_col not in df.columns:
            return pd.DataFrame(columns=df.columns)
        # parse to datetime (coerce errors) then filter by requested inclusive date range
        df[created_at_col] = pd.to_datetime(df[created_at_col], errors='coerce', utc=True)
        return df[(df[created_at_col] >= start_ts) & (df[created_at_col] < end_ts)]

    collected: List[pd.DataFrame] = []
    wrote_header = False
    if save_path:
        # ensure directory exists and start from a clean output file so reruns do not append duplicates
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        if os.path.exists(save_path):
            os.remove(save_path)

    for fpath in csv_files:
        try:
            for chunk in pd.read_csv(fpath, chunksize=chunksize, dtype=str, keep_default_na=False):
                # convert only the relevant column to datetime inside filter
                filtered = _filter_chunk(chunk)
                if filtered.empty:
                    continue
                if save_path:
                    # stream to disk to avoid memory pressure
                    filtered.to_csv(save_path, mode='a', header=not wrote_header, index=False)
                    wrote_header = True
                else:
                    collected.append(filtered)
        except pd.errors.EmptyDataError:
            continue
        except Exception as e:
            # continue on individual file errors but notify user by printing a warning
            print(f'Warning: failed to read {fpath}: {e}')
            continue

    if save_path:
        # when saved to disk, return empty DataFrame (file contains combined results)
        return pd.DataFrame()

    if not collected:
        return pd.DataFrame()

    # combine and ensure created_at is datetime and set UTC tz if present
    combined = pd.concat(collected, ignore_index=True)
    if created_at_col in combined.columns:
        combined[created_at_col] = pd.to_datetime(combined[created_at_col], errors='coerce', utc=True)
    return combined

In [3]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/FB_2019_2022'
# load into memory (can be large)
start_date = '2019-01-01'
end_date = '2022-12-31'
# df_range = load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/FB_2019-01-01_to_2022-12-31.csv'
load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_range = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_date_range_data_from_dir` defined. Use as shown in comments.')

Utility `load_date_range_data_from_dir` defined. Use as shown in comments.


In [4]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/AMZN2019-2022'
# load into memory (can be large)
start_date = '2019-01-01'
end_date = '2022-12-31'
# df_range = load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/AMZN_2019-01-01_to_2022-12-31.csv'
load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_range = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_date_range_data_from_dir` defined. Use as shown in comments.')

Utility `load_date_range_data_from_dir` defined. Use as shown in comments.


In [5]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/NVDA_2013_2022'
# load into memory (can be large)
start_date = '2019-01-01'
end_date = '2022-12-31'
# df_range = load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/NVDA_2019-01-01_to_2022-12-31.csv'
load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_range = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_date_range_data_from_dir` defined. Use as shown in comments.')

Utility `load_date_range_data_from_dir` defined. Use as shown in comments.


In [6]:
# Stream AAPL StockTwits rows for the 2020-2022 experiment period.
root = '../data/StockTwits_2020_2022_Raw/AAPL_2020_2022'
start_date = '2020-01-01'
end_date = '2022-12-31'
out_csv = '../data/Sentiment_Year_Data/AAPL_2020-01-01_to_2022-12-31.csv'
load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date, save_path=out_csv)
print(f'Saved AAPL rows to {out_csv}')

Saved AAPL rows to ../data/Sentiment_Year_Data/AAPL_2020-01-01_to_2022-12-31.csv


In [7]:
# Stream TSLA StockTwits rows for the 2020-2022 experiment period.
root = '../data/StockTwits_2020_2022_Raw/TSLA_2020_2022'
start_date = '2020-01-01'
end_date = '2022-12-31'
out_csv = '../data/Sentiment_Year_Data/TSLA_2020-01-01_to_2022-12-31.csv'
load_date_range_data_from_dir(root, start_date=start_date, end_date=end_date, save_path=out_csv)
print(f'Saved TSLA rows to {out_csv}')

Saved TSLA rows to ../data/Sentiment_Year_Data/TSLA_2020-01-01_to_2022-12-31.csv
